#ANOVASK FULL CODE

In [ ]:
'''
Developed by: Luis Alves Correia Filho

Code integration and refactoring for Version 2.0 assisted by Google Gemini.

If you find this implementation useful for your work, please cite the following publication:

Novo, M. S., Correia Filho, L. A., Ottoni, A. L., Schösser, A., Schulz, P., & Fettweis, G. (2026). Random forest for quality of service prediction in vehicular communication: A statistical approach to hyperparameter tuning. Engineering Applications of Artificial Intelligence, 181, 115754.

'''

# Input Data

## Berlin V2X - Data and Preprocessing

### Data Loading - Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
cell_df = pd.read_parquet('/content/drive/MyDrive/Datasets/Berlin_v2x/cellular_dataframe.parquet')

### Preprocessing, Splitting, and Filtering - Uniform Environment Split

In [ ]:
# Preprocess the data

print(cell_df.shape)  # Shape of the Original Dataset

# Convert the datarate to Mbps
cell_df['datarate'] = cell_df['datarate'] / 1e6

# Filter for downlink and use datarate as qos measurement - the original dataset has downlink and uplink for direction, and for measured_qos has 'datarate' and 'delay'
filtered_data = cell_df.query("direction == 'downlink' & measured_qos == 'datarate'")
print(filtered_data.shape)  # Show shape of the Filtered Data

# Remove incomplete measurements without datarate
filtered_data = filtered_data.dropna(subset='datarate')
print(filtered_data.shape)  # Show shape of the Filtered Data

# Remove stationary measurements (speed=0)
filtered_data = filtered_data.query("speed_kmh != 0")
print(filtered_data.shape)  # Show shape of the Filtered Data

# Remove incomplete measurements without speed (speed=NaN)
filtered_data = filtered_data.dropna(subset='speed_kmh')
print(filtered_data.shape)  # Show shape of the Filtered Data

# Filter for A3D scenario - just to confirm, but the filters above already did that
filtered_data = filtered_data.query("scenario == 'A3D'")
print(filtered_data.shape)  # Show shape of the Filtered Data

# Split data along environments

data_avenue = filtered_data.query("area == 'Avenue'")
print(data_avenue.shape)
data_park = filtered_data.query("area == 'Park'")
print(data_park.shape)
data_residential = filtered_data.query("area == 'Residential'")
print(data_residential.shape)
data_highway = filtered_data.query("area == 'Highway'")
print(data_highway.shape)
data_tunnel = filtered_data.query("area == 'Tunnel'")
print(data_tunnel.shape)

# Feature Selection
qos_column = 'datarate'

feature_columns = [
    # PHY features
    'PCell_RSRP_max',
    'PCell_RSRQ_max',
    'PCell_RSSI_max',
    'PCell_SNR_1',
    'PCell_SNR_2',
    'SCell_RSRP_max',
    'SCell_RSRQ_max',
    'SCell_RSSI_max',
    'SCell_SNR_1',
    'SCell_SNR_2',

    # Physical Shared Channels and Radio Resource Control (RRC)
    'PCell_Downlink_Num_RBs',
    'PCell_Downlink_TB_Size',
    'PCell_Downlink_Average_MCS',
    'PCell_Downlink_bandwidth_MHz',
    'PCell_Cell_Identity',
    'PCell_freq_MHz',
    'SCell_Downlink_Num_RBs',
    'SCell_Downlink_TB_Size',
    'SCell_Downlink_Average_MCS',
    'SCell_Downlink_bandwidth_MHz',
    'SCell_Cell_Identity',
    'SCell_freq_MHz'
]

used_features = len(feature_columns)
print(f"The implementation uses {used_features} features")

# Filter for each environment

x_avenue, y_avenue = data_avenue[feature_columns], data_avenue[qos_column]
x_park, y_park = data_park[feature_columns], data_park[qos_column]
x_residential, y_residential = data_residential[feature_columns], data_residential[qos_column]
x_highway, y_highway = data_highway[feature_columns], data_highway[qos_column]
x_tunnel, y_tunnel = data_tunnel[feature_columns], data_tunnel[qos_column]

x_avenue = x_avenue.fillna(0)
x_park = x_park.fillna(0)
x_residential = x_residential.fillna(0)
x_highway = x_highway.fillna(0)
x_tunnel = x_tunnel.fillna(0)

# Separating data into Training, Validation, and Test
from sklearn.model_selection import train_test_split
train_ratio = 0.80
validation_ratio = 0.10
test_ratio = 0.10

# Avenue
x_traina, x_testav, y_traina, y_testav = train_test_split(x_avenue, y_avenue, test_size=1 - train_ratio, random_state=0)
x_vala, x_testa, y_vala, y_testa = train_test_split(x_testav, y_testav, test_size=test_ratio / (test_ratio + validation_ratio), random_state=0)

# Residential
x_trainr, x_testrv, y_trainr, y_testrv = train_test_split(x_residential, y_residential, test_size=1 - train_ratio, random_state=0)
x_valr, x_testr, y_valr, y_testr = train_test_split(x_testrv, y_testrv, test_size=test_ratio / (test_ratio + validation_ratio), random_state=0)

# Park
x_trainp, x_testpv, y_trainp, y_testpv = train_test_split(x_park, y_park, test_size=1 - train_ratio, random_state=0)
x_valp, x_testp, y_valp, y_testp = train_test_split(x_testpv, y_testpv, test_size=test_ratio / (test_ratio + validation_ratio), random_state=0)

# Highway
x_trainh, x_testhv, y_trainh, y_testhv = train_test_split(x_highway, y_highway, test_size=1 - train_ratio, random_state=0)
x_valh, x_testh, y_valh, y_testh = train_test_split(x_testhv, y_testhv, test_size=test_ratio / (test_ratio + validation_ratio), random_state=0)

# Tunnel
x_traint, x_testtv, y_traint, y_testtv = train_test_split(x_tunnel, y_tunnel, test_size=1 - train_ratio, random_state=0)
x_valt, x_testt, y_valt, y_testt = train_test_split(x_testtv, y_testtv, test_size=test_ratio / (test_ratio + validation_ratio), random_state=0)

# Total data - separated per environment
x_train = pd.concat([x_traina, x_trainp, x_trainr, x_trainh, x_traint])
print(x_train.shape)
x_val = pd.concat([x_vala, x_valp, x_valr, x_valh, x_valt])
print(x_val.shape)
x_test = pd.concat([x_testa, x_testp, x_testr, x_testh, x_testt])
print(x_test.shape)
y_train = pd.concat([y_traina, y_trainp, y_trainr, y_trainh, y_traint])
y_val = pd.concat([y_vala, y_valp, y_valr, y_valh, y_valt])
y_test = pd.concat([y_testa, y_testp, y_testr, y_testh, y_testt])

(207434, 159)
(65802, 159)
(52673, 159)
(32180, 159)
(32060, 159)
(32060, 159)
(7066, 159)
(11698, 159)
(8580, 159)
(4527, 159)
(189, 159)
The implementation uses 22 features


/tmp/ipykernel_4897/2989840376.py:87: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  x_tunnel = x_tunnel.fillna(0)


(25646, 22)
(3207, 22)
(3207, 22)


# Part 1 - Isolated Hyperparameters

In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

In [ ]:
# =============================================================================
# Part 1 - Isolated Hyperparameters
# =============================================================================


#Search space

ndepth=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20]# number of depth
nestimators=[50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 800, 850, 900, 950, 1000]# number of estimators
maxfeatures = ["log2", "sqrt", 5, 6, int(used_features / 3), 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]  # maximum features
maxsamples = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]  # maximum number of samples to train each tree
maxleafn=[50, 100, 200, 300, 400, 500, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000, 6000, 7000, 8000, 9000, 10000, None]# maximum leaf nodes
minsamplesleaf=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]# minimum number of samples required to be at a leaf node
minsamplessplit=[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]# minimum number of samples required to split an internal node
criteri = ["squared_error", "absolute_error", "friedman_mse", "poisson"]


In [ ]:
reps = 5  # Number of repetitions per hyperparameter

# Global storage structures
impacto_hiperparametros = {}
resumo_parte1 = []


def registrar_impacto(
    nome_parametro,
    lista_valores,
    matriz_rmse,
    matriz_mae,
    matriz_r2,
    matriz_ttreino,
    matriz_tpred,
):
  """Calculates global statistics and selects the best RMSE value per quartile."""
  medias_rmse = np.mean(matriz_rmse, axis=1)
  medias_mae = np.mean(matriz_mae, axis=1)
  medias_r2 = np.mean(matriz_r2, axis=1)
  medias_ttreino = np.mean(matriz_ttreino, axis=1)
  medias_tpred = np.mean(matriz_tpred, axis=1)

  # Selection of the lowest RMSE for each of the 4 interval quartiles
  indices_quartis = np.array_split(range(len(lista_valores)), 4)
  melhores_4_quartis = []

  for indices in indices_quartis:
    if len(indices) > 0:
      rmse_quartil = medias_rmse[indices]
      idx_local = np.argmin(rmse_quartil)
      idx_global = indices[idx_local]
      melhores_4_quartis.append(lista_valores[idx_global])

  # Saves global statistics for the dynamic transition to Part 2
  impacto_hiperparametros[nome_parametro] = {
      'diff_rmse': np.max(medias_rmse) - np.min(medias_rmse),
      'melhor_valor_rmse': lista_valores[np.argmin(medias_rmse)],
      'diff_mae': np.max(medias_mae) - np.min(medias_mae),
      'melhor_valor_mae': lista_valores[np.argmin(medias_mae)],
      'diff_r2': np.max(medias_r2) - np.min(medias_r2),
      'melhor_valor_r2': lista_valores[np.argmax(medias_r2)],
      'diff_ttreino': np.max(medias_ttreino) - np.min(medias_ttreino),
      'melhor_valor_ttreino': lista_valores[np.argmin(medias_ttreino)],
      'diff_tpred': np.max(medias_tpred) - np.min(medias_tpred),
      'melhor_valor_tpred': lista_valores[np.argmin(medias_tpred)],
      'melhores_4_quartis': melhores_4_quartis,
  }

  # Saves the summary of best/worst RMSE for CSV export
  idx_melhor_rmse = np.argmin(medias_rmse)
  idx_pior_rmse = np.argmax(medias_rmse)
  resumo_parte1.append({
      'Hyperparameter': nome_parametro,
      'Best Value': lista_valores[idx_melhor_rmse],
      'Best RMSE': medias_rmse[idx_melhor_rmse],
      'Worst Value': lista_valores[idx_pior_rmse],
      'Worst RMSE': medias_rmse[idx_pior_rmse],
      'RMSE Difference (Worst - Best)': (
          medias_rmse[idx_pior_rmse] - medias_rmse[idx_melhor_rmse]
      ),
  })

In [ ]:
# -----------------------------------------------------------------------------
# 2. Study of Each Hyperparameter
# -----------------------------------------------------------------------------

# --- Study Number of Depth ---
print('\n=== Starting study: max_depth ===')
shapetest = (len(ndepth), reps)
r2_scorerf = np.zeros(shapetest)
MAErf = np.zeros(shapetest)
RMSErf = np.zeros(shapetest)
temptrain = np.zeros(shapetest)
temppred = np.zeros(shapetest)

for j in range(reps):
  print(f'Repetition: {j+1}')
  for i, ndep in enumerate(ndepth):
    rf = RandomForestRegressor(
        max_depth=ndep, max_features=int(used_features / 3)
    )

    t_init = time.time()
    rf.fit(x_train, y_train)
    temptrain[i, j] = time.time() - t_init

    t_inip = time.time()
    y_pred = rf.predict(x_train)
    temppred[i, j] = time.time() - t_inip

    r2_scorerf[i, j] = r2_score(y_train, y_pred)
    MAErf[i, j] = mean_absolute_error(y_train, y_pred)
    RMSErf[i, j] = root_mean_squared_error(y_train, y_pred)

registrar_impacto(
    'max_depth', ndepth, RMSErf, MAErf, r2_scorerf, temptrain, temppred
)


=== Starting study: max_depth ===
Repetition: 1
Repetition: 2
Repetition: 3
Repetition: 4
Repetition: 5


In [ ]:
# --- Study Number of Estimators ---
print('\n=== Starting study: n_estimators ===')
shapetest = (len(nestimators), reps)
r2_scorerf = np.zeros(shapetest)
MAErf = np.zeros(shapetest)
RMSErf = np.zeros(shapetest)
temptrain = np.zeros(shapetest)
temppred = np.zeros(shapetest)

for j in range(reps):
  print(f'Repetition: {j+1}')
  for i, nest in enumerate(nestimators):
    rf = RandomForestRegressor(
        n_estimators=nest, max_features=int(used_features / 3)
    )

    t_init = time.time()
    rf.fit(x_train, y_train)
    temptrain[i, j] = time.time() - t_init

    t_inip = time.time()
    y_pred = rf.predict(x_train)
    temppred[i, j] = time.time() - t_inip

    r2_scorerf[i, j] = r2_score(y_train, y_pred)
    MAErf[i, j] = mean_absolute_error(y_train, y_pred)
    RMSErf[i, j] = root_mean_squared_error(y_train, y_pred)

registrar_impacto(
    'n_estimators', nestimators, RMSErf, MAErf, r2_scorerf, temptrain, temppred
)


=== Starting study: n_estimators ===
Repetition: 1
Repetition: 2
Repetition: 3
Repetition: 4
Repetition: 5


In [ ]:
# --- Study Max Features ---
print('\n=== Starting study: max_features ===')
shapetest = (len(maxfeatures), reps)
r2_scorerf = np.zeros(shapetest)
MAErf = np.zeros(shapetest)
RMSErf = np.zeros(shapetest)
temptrain = np.zeros(shapetest)
temppred = np.zeros(shapetest)

for j in range(reps):
  print(f'Repetition: {j+1}')
  for i, max_f in enumerate(maxfeatures):
    rf = RandomForestRegressor(max_features=max_f)

    t_init = time.time()
    rf.fit(x_train, y_train)
    temptrain[i, j] = time.time() - t_init

    t_inip = time.time()
    y_pred = rf.predict(x_train)
    temppred[i, j] = time.time() - t_inip

    r2_scorerf[i, j] = r2_score(y_train, y_pred)
    MAErf[i, j] = mean_absolute_error(y_train, y_pred)
    RMSErf[i, j] = root_mean_squared_error(y_train, y_pred)

registrar_impacto(
    'max_features', maxfeatures, RMSErf, MAErf, r2_scorerf, temptrain, temppred
)


=== Starting study: max_features ===
Repetition: 1
Repetition: 2
Repetition: 3
Repetition: 4
Repetition: 5


In [ ]:
# --- Study Max Samples ---
print('\n=== Starting study: max_samples ===')
shapetest = (len(maxsamples), reps)
r2_scorerf = np.zeros(shapetest)
MAErf = np.zeros(shapetest)
RMSErf = np.zeros(shapetest)
temptrain = np.zeros(shapetest)
temppred = np.zeros(shapetest)

for j in range(reps):
  print(f'Repetition: {j+1}')
  for i, maxs in enumerate(maxsamples):
    rf = RandomForestRegressor(
        max_samples=maxs, max_features=int(used_features / 3)
    )

    t_init = time.time()
    rf.fit(x_train, y_train)
    temptrain[i, j] = time.time() - t_init

    t_inip = time.time()
    y_pred = rf.predict(x_train)
    temppred[i, j] = time.time() - t_inip

    r2_scorerf[i, j] = r2_score(y_train, y_pred)
    MAErf[i, j] = mean_absolute_error(y_train, y_pred)
    RMSErf[i, j] = root_mean_squared_error(y_train, y_pred)

registrar_impacto(
    'max_samples', maxsamples, RMSErf, MAErf, r2_scorerf, temptrain, temppred
)


=== Starting study: max_samples ===
Repetition: 1
Repetition: 2
Repetition: 3
Repetition: 4
Repetition: 5


In [ ]:
# --- Study Max Leaf Nodes ---
print('\n=== Starting study: max_leaf_nodes ===')
shapetest = (len(maxleafn), reps)
r2_scorerf = np.zeros(shapetest)
MAErf = np.zeros(shapetest)
RMSErf = np.zeros(shapetest)
temptrain = np.zeros(shapetest)
temppred = np.zeros(shapetest)

for j in range(reps):
  print(f'Repetition: {j+1}')
  for i, maxl in enumerate(maxleafn):
    rf = RandomForestRegressor(
        max_leaf_nodes=maxl, max_features=int(used_features / 3)
    )

    t_init = time.time()
    rf.fit(x_train, y_train)
    temptrain[i, j] = time.time() - t_init

    t_inip = time.time()
    y_pred = rf.predict(x_train)
    temppred[i, j] = time.time() - t_inip

    r2_scorerf[i, j] = r2_score(y_train, y_pred)
    MAErf[i, j] = mean_absolute_error(y_train, y_pred)
    RMSErf[i, j] = root_mean_squared_error(y_train, y_pred)

registrar_impacto(
    'max_leaf_nodes', maxleafn, RMSErf, MAErf, r2_scorerf, temptrain, temppred
)


=== Starting study: max_leaf_nodes ===
Repetition: 1
Repetition: 2
Repetition: 3
Repetition: 4
Repetition: 5


In [ ]:
# --- Study Min Samples Leaf ---
print('\n=== Starting study: min_samples_leaf ===')
shapetest = (len(minsamplesleaf), reps)
r2_scorerf = np.zeros(shapetest)
MAErf = np.zeros(shapetest)
RMSErf = np.zeros(shapetest)
temptrain = np.zeros(shapetest)
temppred = np.zeros(shapetest)

for j in range(reps):
  print(f'Repetition: {j+1}')
  for i, minsamples in enumerate(minsamplesleaf):
    rf = RandomForestRegressor(
        min_samples_leaf=minsamples, max_features=int(used_features / 3)
    )

    t_init = time.time()
    rf.fit(x_train, y_train)
    temptrain[i, j] = time.time() - t_init

    t_inip = time.time()
    y_pred = rf.predict(x_train)
    temppred[i, j] = time.time() - t_inip

    r2_scorerf[i, j] = r2_score(y_train, y_pred)
    MAErf[i, j] = mean_absolute_error(y_train, y_pred)
    RMSErf[i, j] = root_mean_squared_error(y_train, y_pred)

registrar_impacto(
    'min_samples_leaf',
    minsamplesleaf,
    RMSErf,
    MAErf,
    r2_scorerf,
    temptrain,
    temppred,
)


=== Starting study: min_samples_leaf ===
Repetition: 1
Repetition: 2
Repetition: 3
Repetition: 4
Repetition: 5


In [ ]:
# --- Study Min Samples Split ---
print('\n=== Starting study: min_samples_split ===')
shapetest = (len(minsamplessplit), reps)
r2_scorerf = np.zeros(shapetest)
MAErf = np.zeros(shapetest)
RMSErf = np.zeros(shapetest)
temptrain = np.zeros(shapetest)
temppred = np.zeros(shapetest)

for j in range(reps):
  print(f'Repetition: {j+1}')
  for i, minss in enumerate(minsamplessplit):
    rf = RandomForestRegressor(
        min_samples_split=minss, max_features=int(used_features / 3)
    )

    t_init = time.time()
    rf.fit(x_train, y_train)
    temptrain[i, j] = time.time() - t_init

    t_inip = time.time()
    y_pred = rf.predict(x_train)
    temppred[i, j] = time.time() - t_inip

    r2_scorerf[i, j] = r2_score(y_train, y_pred)
    MAErf[i, j] = mean_absolute_error(y_train, y_pred)
    RMSErf[i, j] = root_mean_squared_error(y_train, y_pred)

registrar_impacto(
    'min_samples_split',
    minsamplessplit,
    RMSErf,
    MAErf,
    r2_scorerf,
    temptrain,
    temppred,
)


=== Starting study: min_samples_split ===
Repetition: 1
Repetition: 2
Repetition: 3
Repetition: 4
Repetition: 5


In [ ]:
# --- Study Criterion ---
print('\n=== Starting study: criterion ===')
shapetest = (len(criteri), reps)
r2_scorerf = np.zeros(shapetest)
MAErf = np.zeros(shapetest)
RMSErf = np.zeros(shapetest)
temptrain = np.zeros(shapetest)
temppred = np.zeros(shapetest)

for j in range(reps):
  print(f'Repetition: {j+1}')
  for i, cr in enumerate(criteri):
    rf = RandomForestRegressor(
        criterion=cr, max_features=int(used_features / 3)
    )

    t_init = time.time()
    rf.fit(x_train, y_train)
    temptrain[i, j] = time.time() - t_init

    t_inip = time.time()
    y_pred = rf.predict(x_train)
    temppred[i, j] = time.time() - t_inip

    r2_scorerf[i, j] = r2_score(y_train, y_pred)
    MAErf[i, j] = mean_absolute_error(y_train, y_pred)
    RMSErf[i, j] = root_mean_squared_error(y_train, y_pred)

registrar_impacto(
    'criterion', criteri, RMSErf, MAErf, r2_scorerf, temptrain, temppred
)


=== Starting study: criterion ===
Repetition: 1
Repetition: 2
Repetition: 3
Repetition: 4
Repetition: 5


In [ ]:
# -----------------------------------------------------------------------------
# 3. Export Summary of Part 1
# -----------------------------------------------------------------------------
df_parte1 = pd.DataFrame(resumo_parte1)
df_parte1.to_csv('resultado_parte1.csv', sep=';', decimal=',', index=False)

print('\n=== PART 1 COMPLETED AND SAVED TO resultado_parte1.csv ===')
print(df_parte1)

# 2. Saving in Google Drive (root)
caminho_drive = '/content/drive/MyDrive/resultado_parte1.csv'
df_parte1.to_csv(caminho_drive, sep=';', decimal=',', index=False)


=== PART 1 COMPLETED AND SAVED TO resultado_parte1.csv ===
      Hyperparameter Best Value  Best RMSE     Worst Value  Worst RMSE  \
0          max_depth         10   6.874344               1   23.813528   
1       n_estimators        200   3.435257              50    3.572510   
2       max_features          8   3.467261              22    3.558423   
3        max_samples        1.0   3.476949            0.05    9.252984   
4     max_leaf_nodes       None   3.472828              50    9.523700   
5   min_samples_leaf          1   3.478106              10    7.669387   
6  min_samples_split          2   3.447235              10    5.173151   
7          criterion    poisson   3.474299  absolute_error    3.503413   

   RMSE Difference (Worst - Best)  
0                       16.939184  
1                        0.137253  
2                        0.091162  
3                        5.776035  
4                        6.050872  
5                        4.191281  
6                    

# Part 2 - Hyperparameter Combinations

In [ ]:
# =============================================================================
# Part 2 - Hyperparameter Combinations for RandomForest Algorithm
# =============================================================================
import itertools
import pandas as pd
import time

print("\n=== HYPERPARAMETER IMPACT (RMSE, MAE, R2, and Times) ===")

# 1. Print complete impact report in the terminal
for hp, dados in impacto_hiperparametros.items():
    print(
        f"{hp}:\n"
        f"  - RMSE: Var = {dados['diff_rmse']:.4f} (Best: {dados['melhor_valor_rmse']})\n"
        f"  - MAE : Var = {dados['diff_mae']:.4f} (Best: {dados['melhor_valor_mae']})\n"
        f"  - R2  : Var = {dados['diff_r2']:.4f} (Best: {dados['melhor_valor_r2']})\n"
        f"  - Train Time: Var = {dados['diff_ttreino']:.4f}s (Fastest: {dados['melhor_valor_ttreino']})\n"
        f"  - Pred Time : Var = {dados['diff_tpred']:.4f}s (Fastest: {dados['melhor_valor_tpred']})\n"
    )

# 2. Sort and select exclusively by largest RMSE fluctuation (impact)
hps_ordenados = sorted(impacto_hiperparametros.items(), key=lambda item: item[1]['diff_rmse'], reverse=True)

# Separate Top 3 (combination study) and Fixed parameters
top3 = hps_ordenados[:3]
fixos = hps_ordenados[3:]

nomes_top3 = [hp[0] for hp in top3]

# Use global best RMSE value to fix parameters not included in combination study
parametros_fixos = {hp[0]: hp[1]['melhor_valor_rmse'] for hp in fixos}

print(f"\nTop 3 Variables for combinations study (based on RMSE): {nomes_top3}")
print(f"Fixed Parameters: {parametros_fixos}\n")

# 3. Dynamically retrieve the 4 winning values (1 per quartile) for the Top 3
listas_top3 = [impacto_hiperparametros[nome]['melhores_4_quartis'] for nome in nomes_top3]

print("=== VALUES SELECTED PER QUARTILE (LOWEST RMSE) FOR THE STUDY ===")
print(f"{nomes_top3[0]}: {listas_top3[0]}")
print(f"{nomes_top3[1]}: {listas_top3[1]}")
print(f"{nomes_top3[2]}: {listas_top3[2]}\n")

# Automatically generate all 64 possible combinations (4 x 4 x 4)
combinacoes = list(itertools.product(*listas_top3))

resultados_parte2 = []
reps = 5

# 4. Execute Combination Study
for j in range(reps):
    print(f'\nRepetition: {j+1}')
    comb_id = 1

    for valores_comb in combinacoes:
        # Build dictionary with fixed parameters
        args_rf = parametros_fixos.copy()

        # Inject the 3 variable parameters for this combination
        args_rf[nomes_top3[0]] = valores_comb[0]
        args_rf[nomes_top3[1]] = valores_comb[1]
        args_rf[nomes_top3[2]] = valores_comb[2]

        # Instantiate model by unpacking arguments directly
        rf = RandomForestRegressor(**args_rf)

        print(f"Training Comb {comb_id}: {nomes_top3[0]}={valores_comb[0]} | {nomes_top3[1]}={valores_comb[1]} | {nomes_top3[2]}={valores_comb[2]}")

        # Training
        tempo_init = time.time()
        rf.fit(x_train, y_train)
        tempo_treino = time.time() - tempo_init

        # Prediction
        tempo_inip = time.time()
        y_pred = rf.predict(x_train)
        tempo_pred = time.time() - tempo_inip

        # Metrics Extraction
        r2 = r2_score(y_train, y_pred)
        mae = mean_absolute_error(y_train, y_pred)
        rmse = root_mean_squared_error(y_train, y_pred)

        # Structured row record
        linha = {
          "Comb": comb_id,
          nomes_top3[0]: valores_comb[0],
          nomes_top3[1]: valores_comb[1],
          nomes_top3[2]: valores_comb[2],
          "Repetition": j + 1,
          "yr2": r2,
          "ymae": mae,
          "yrmse": rmse,
          "ytt": tempo_treino,
          "ytp": tempo_pred
        }
        resultados_parte2.append(linha)
        comb_id += 1

# =============================================================================
# 5. Completion and Export
# =============================================================================

# Convert to DataFrame and sort numerically by Combination and Repetition
df_parte2 = pd.DataFrame(resultados_parte2).sort_values(by=["Comb", "Repetition"])

# Export raw results formatted with tab-delimited columns for R reading
df_parte2.to_csv("resultado_parte2.csv", sep="\t", decimal=".", index=False)

# Export results for Google Drive
df_parte2.to_csv("/content/drive/MyDrive/resultado_parte2.csv", sep="\t", decimal=".", index=False)


print("\n=== STATISTICS PER COMBINATION (PART 2) ===")

# Dynamically group using the names of the 3 parameters to compute means and standard deviations
estatisticas_comb = df_parte2.groupby(
    ["Comb", nomes_top3[0], nomes_top3[1], nomes_top3[2]]
).agg(
    MAE_Media=('ymae', 'mean'),
    MAE_Desvio=('ymae', 'std'),
    RMSE_Media=('yrmse', 'mean'),
    RMSE_Desvio=('yrmse', 'std'),
    R2_Media=('yr2', 'mean'),
    R2_Desvio=('yr2', 'std'),
    Treino_Tempo_Media=('ytt', 'mean'),
    Pred_Tempo_Media=('ytp', 'mean')
).reset_index()

# Display formatted and organized metrics in terminal
for index, row in estatisticas_comb.iterrows():
    print(
        f"Comb {int(row['Comb'])} "
        f"[{nomes_top3[0]}: {row[nomes_top3[0]]}, {nomes_top3[1]}: {row[nomes_top3[1]]}, {nomes_top3[2]}: {row[nomes_top3[2]]}]:\n"
        f"  - MAE:  {row['MAE_Media']:.4f} (+/- {row['MAE_Desvio']:.4f})\n"
        f"  - RMSE: {row['RMSE_Media']:.4f} (+/- {row['RMSE_Desvio']:.4f})\n"
        f"  - R2:   {row['R2_Media']:.4f} (+/- {row['R2_Desvio']:.4f})\n"
        f"  - Train Time: {row['Treino_Tempo_Media']:.4f}s | Pred Time: {row['Pred_Tempo_Media']:.4f}s\n"
    )


=== HYPERPARAMETER IMPACT (RMSE, MAE, R2, and Times) ===
max_depth:
  - RMSE: Var = 16.9392 (Best: 10)
  - MAE : Var = 12.7110 (Best: 10)
  - R2  : Var = 0.3507 (Best: 10)
  - Train Time: Var = 6.9206s (Fastest: 1)
  - Pred Time : Var = 0.1681s (Fastest: 1)

n_estimators:
  - RMSE: Var = 0.1373 (Best: 200)
  - MAE : Var = 0.0780 (Best: 200)
  - R2  : Var = 0.0006 (Best: 200)
  - Train Time: Var = 20.0373s (Fastest: 50)
  - Pred Time : Var = 0.9226s (Fastest: 50)

max_features:
  - RMSE: Var = 0.0912 (Best: 8)
  - MAE : Var = 0.0514 (Best: 8)
  - R2  : Var = 0.0004 (Best: 8)
  - Train Time: Var = 34.6711s (Fastest: log2)
  - Pred Time : Var = 0.0788s (Fastest: 7)

max_samples:
  - RMSE: Var = 5.7760 (Best: 1.0)
  - MAE : Var = 3.3696 (Best: 1.0)
  - R2  : Var = 0.0496 (Best: 1.0)
  - Train Time: Var = 12.6613s (Fastest: 0.05)
  - Pred Time : Var = 0.4064s (Fastest: 0.05)

max_leaf_nodes:
  - RMSE: Var = 6.0509 (Best: None)
  - MAE : Var = 3.9286 (Best: None)
  - R2  : Var = 0.0530 (Bes

# Part 3 - ANOVA + SCOTT-KNOTT

In this section, analysis of variance is performed along with the application of the Scott-Knott algorithm to define groups and determine the best hyperparameter combination.

In [ ]:
#This code was adapted from the original R implementation by André L. C. Ottoni.

# =============================================================================
# Part 3 - Setup and Data Transfer to R (rpy2)
# =============================================================================
import pandas as pd
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri

pandas2ri.activate()

# 1. Read Part 2 results directly from Google Drive CSV file
caminho_drive_p2 = '/content/drive/MyDrive/resultado_parte2.csv'
df_parte2 = pd.read_csv(caminho_drive_p2, sep='\t', decimal='.')

# 2. Dynamically identify the top 3 hyperparameter names from CSV columns
cols_hp = list(df_parte2.columns)[1:4]
nomes_top3 = cols_hp

# 3. Installation of ScottKnott R package (Drive local file or CRAN fallback)
caminho_tar_sk = "/content/drive/MyDrive/Doutorado - UFBA/Códigos de Programação/Novo Código em Python - ANOVASK/analise0407-arquivos/ScottKnott_1.2-7.tar.gz"
try:
  robjects.r(
      f'install.packages("{caminho_tar_sk}", repos = NULL, type = "source")'
  )
except Exception as e:
  print(f"Local package file not found ({e}). Attempting CRAN installation...")
  try:
    robjects.r('install.packages("ScottKnott", repos = "https://cloud.r-project.org")')
  except Exception as err:
    print(f"CRAN installation note: {err}")

# 4. Transfer DataFrame and hyperparameter names to R session
robjects.globalenv['dados'] = pandas2ri.py2rpy(df_parte2)
robjects.globalenv['hp1_name'] = nomes_top3[0]
robjects.globalenv['hp2_name'] = nomes_top3[1]
robjects.globalenv['hp3_name'] = nomes_top3[2]

print(f"Data successfully loaded from {caminho_drive_p2} and transferred to R session!")
print(f"Detected Top 3 Hyperparameters: {nomes_top3}")

(as ‘lib’ is unspecified)



Data successfully loaded from /content/drive/MyDrive/resultado_parte2.csv and transferred to R session!
Detected Top 3 Hyperparameters: ['max_depth', 'max_leaf_nodes', 'max_samples']


In [ ]:
# Commented out IPython magic to ensure Python compatibility.
# %load_ext rpy2.ipython

In [ ]:
%%R

library(ScottKnott)

# Retrieve dynamic names of the 3 selected hyperparameters
hp1 <<- hp1_name
hp2 <<- hp2_name
hp3 <<- hp3_name

# Build statistical formulas dynamically for ANOVA
form1 <- as.formula(paste("yrmse ~ factor(", hp1, ") * factor(", hp2, ") * factor(", hp3, ")"))
form2 <- as.formula(paste("yrmse ~ factor(", hp2, ") * factor(", hp1, ") * factor(", hp3, ")"))
form3 <- as.formula(paste("yrmse ~ factor(", hp3, ") * factor(", hp2, ") * factor(", hp1, ")"))

# ANOVA Models
modelo  <- aov(yrmse ~ factor(Comb), data = dados)
modelo1 <- aov(form1, data = dados)
modelo2 <- aov(form2, data = dados)
modelo3 <- aov(form3, data = dados)

cat("\n=================== MODEL INFORMATION (ANOVA) ===================\n")
print(summary(modelo))
print(summary(modelo1))
print(summary(modelo2))
print(summary(modelo3))

cat("\n=================== NORMALITY TEST (KS-TEST) ===================\n")
print(ks.test(resid(modelo), 'pnorm', mean(resid(modelo)), sd(resid(modelo))))
print(ks.test(resid(modelo1), 'pnorm', mean(resid(modelo1)), sd(resid(modelo1))))
print(ks.test(resid(modelo2), 'pnorm', mean(resid(modelo2)), sd(resid(modelo2))))
print(ks.test(resid(modelo3), 'pnorm', mean(resid(modelo3)), sd(resid(modelo3))))

cat("\n=================== HOMOGENEITY TEST (BARTLETT) ===================\n")
print(bartlett.test(yrmse ~ factor(Comb), data = dados))

# Execution of Scott-Knott objects in global environment (<<-)
sk  <<- SK(modelo)   # General combinations
sk1 <<- SK(modelo1)  # Top 1 HP
sk2 <<- SK(modelo2)  # Top 2 HP
sk3 <<- SK(modelo3)  # Top 3 HP

cat("\n=================== SCOTT-KNOTT SUMMARY ===================\n")
summary(sk)
summary(sk1)
summary(sk2)
summary(sk3)


=================== MODEL INFORMATION (ANOVA) ===================
              Df Sum Sq Mean Sq F value Pr(>F)    
factor(Comb)  63 1952.4  30.991   21325 <2e-16 ***
Residuals    256    0.4   0.001                   
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1
                                                              Df Sum Sq Mean Sq
factor(max_depth)                                              3 1928.3   642.8
factor(max_leaf_nodes)                                         3    3.3     1.1
factor(max_samples)                                            3    5.3     1.8
factor(max_depth):factor(max_leaf_nodes)                       9    8.8     1.0
factor(max_depth):factor(max_samples)                          9    5.6     0.6
factor(max_leaf_nodes):factor(max_samples)                     9    0.2     0.0
factor(max_depth):factor(max_leaf_nodes):factor(max_samples)  27    0.9     0.0
Residuals                                                    256    0.4  

In [ ]:
%%R -w 900 -h 500

par(mfrow = c(1, 1))
plot(sk,
     col = rainbow(max(sk$groups)),
     rl = FALSE,
     id.las = 2,
     main = "Scott-Knott: Combinations",
     xlab = "Combinations",
     ylab = "RMSE (mean)")

In [ ]:
%%R -w 700 -h 450

par(mfrow = c(1, 1))
plot(sk1,
     col = rainbow(max(sk1$groups)),
     rl = FALSE,
     id.las = 2,
     main = paste("Scott-Knott:", hp1),
     xlab = hp1,
     ylab = "RMSE (mean)")

In [ ]:
%%R -w 700 -h 450

par(mfrow = c(1, 1))
plot(sk2,
     col = rainbow(max(sk2$groups)),
     rl = FALSE,
     id.las = 2,
     main = paste("Scott-Knott:", hp2),
     xlab = hp2,
     ylab = "RMSE (mean)")

In [ ]:
%%R -w 700 -h 450

par(mfrow = c(1, 1))
plot(sk3,
     col = rainbow(max(sk3$groups)),
     rl = FALSE,
     id.las = 2,
     main = paste("Scott-Knott:", hp3),
     xlab = hp3,
     ylab = "RMSE (mean)")